# Prefect

This is a tutorial to use the Prefect cluster from Jupyter, without Dask.

In [1]:
import os
print(f"Internal Prefect server: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['PREFECT_PUBLIC']}/dashboard"
print(f"Public Prefect dashboard: {dashboard}")

Internal Prefect server: http://prefect-server:4200/api
Public Prefect dashboard: http://localhost:4200/dashboard


In [2]:
from resources.utils import *
# Init environment before running a demo notebook.
init_demo()

# In local mode, init the prefect blocks.
# NOTE: In the cluster, the blocks must be created only once by the admin.
await init_prefect_blocks()

from resources.utils import *  # reload the global vars again

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000


In [3]:
%%bash
prefect block ls

                                     Blocks                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                ┃ Type            ┃ … ┃ Slug               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ 3698916f-88fe-46fa-bfa6-20e7c408… │ Remote File Sy… │ … │ remote-file-syste… │
└───────────────────────────────────┴─────────────────┴───┴────────────────────┘
                 List Block Types using `prefect block type ls`                 


In [21]:
# Other imports
import getpass
import json
import logging
import os
import prefect
from resources.my_shared_utils import get_ip_address

# When deploying, the prefect flows and tasks must be implemented in a python module.
# We cannot implement them from jupyter cells.
import my_prefect

# Data to test the example flow
my_data = [
    "PrefectHQ/prefect",
    "pydantic/pydantic",
    "huggingface/transformers"
]

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

### Implement the `quickstart` tutorial
See: https://docs.prefect.io/v3/get-started/quickstart

When calling the flow as a normal python function, the flow and tasks are run by Prefect on your local client environment = your Jupyter or terminal.

This is the easiest way to test your Prefect code because the same environment, Python interpreter and files are shared between your client, flow and tasks. But this is less performant because your tasks are not distributed on the cluster.

In [5]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

# Run the flow
my_prefect.flow_show_stars(my_data)

15:34:33.847 | WARNING | root - Client IP address: 172.18.0.21

15:34:33.914 | INFO    | prefect.engine - Created flow run 'vanilla-moose' for flow 'flow-show-stars'

15:34:33.915 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/cc9a3af4-0c85-4f71-b9a5-f2ddefb96d0a

15:34:33.954 | WARNING | Flow run 'vanilla-moose' - Flow IP address: 172.18.0.21

15:34:33.983 | WARNING | Task run 'task_fetch_stats-b90' - 'fetch_stats' task IP address: 172.18.0.21

...

15:34:34.603 | INFO    | Task run 'task_fetch_stats-6e0' - Finished in state Completed()

15:34:34.619 | WARNING | Task run 'task_get_stars-9e8' - 'get_stars' task IP address: 172.18.0.21

15:34:34.622 | INFO    | Task run 'task_get_stars-9e8' - Finished in state Completed()

15:34:34.624 | WARNING | Flow run 'vanilla-moose' - Result for repository 'huggingface/transformers': 137970 stars

15:34:34.677 | INFO    | Flow run 'vanilla-moose' - Finished in state Completed()

<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs above that your client IP address is also used by the Prefect flow and tasks.
  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.

### Run flows in local processes
See: https://docs.prefect.io/v3/deploy/run-flows-in-local-processes

Create a deployment for a flow by calling the `serve` method.

As for the quickstart above, the same environment, Python interpreter and files are shared between your client, flow and tasks.

In [22]:
# Deploy the flow
task = my_prefect.hack_for_jupyter( # we need a hack to deploy from jupyter
    my_prefect.flow_show_stars.serve,
    name="serve-python",
    tags=["tutorial"],
)
name = "flow-show-stars/serve-python"
await my_prefect.wait_for_deployment(name)

Finished deploying prefect flow: 'flow-show-stars/serve-python'
Your flow 'flow-show-stars' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'flow-show-stars/serve-python'

You can also run your flow via the Prefect UI: http://prefect-server:4200/deployments/deployment/67e9eda7-95df-4b30-9c90-613c1624cc4a



In [24]:
%%bash -s "$name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2"

Creating flow run for deployment 'flow-show-stars/serve-python'...
Created flow run 'prophetic-gaur'.
└── UUID: 8b58ac11-ea77-4fd8-bc76-73a74d1c7e20
└── Parameters: {'github_repos': ['PrefectHQ/prefect', 'pydantic/pydantic', 'huggingface/transformers']}
└── Job Variables: {}
└── Scheduled start time: 2025-01-24 16:15:32 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/8b58ac11-ea77-4fd8-bc76-73a74d1c7e20


16:15:36.332 | INFO    | prefect.flow_runs.runner - Runner 'serve-python' submitting flow run '8b58ac11-ea77-4fd8-bc76-73a74d1c7e20'

16:15:36.355 | INFO    | prefect.flow_runs.runner - Opening process...

16:15:36.364 | INFO    | prefect.flow_runs.runner - Completed submission of flow run '8b58ac11-ea77-4fd8-bc76-73a74d1c7e20'

16:15:37.329 | INFO    | Flow run 'prophetic-gaur' - Downloading flow code from storage at '.'
16:15:37.388 | WARNING | Flow run 'prophetic-gaur' - Flow IP address: 172.18.0.21
16:15:37.449 | WARNING | Task run 'task_fetch_stats-bdf' - 'fetch_stats' task IP address: 172.18.0.21
16:15:37.738 | INFO    | Task run 'task_fetch_stats-bdf' - Finished in state Completed()
16:15:37.761 | WARNING | Task run 'task_get_stars-97d' - 'get_stars' task IP address: 172.18.0.21
16:15:37.763 | INFO    | Task run 'task_get_stars-97d' - Finished in state Completed()
16:15:37.764 | WARNING | Flow run 'prophetic-gaur' - Result for repository 'PrefectHQ/prefect': 18110 stars
16:15:37.778 | WARNING | Task run 'task_fetch_stats-b49' - 'fetch_stats' task IP address: 172.18.0.21
16:15:39.054 | INFO    | Task run 'task_fetch_stats-b49' - Finished in state Completed()
16:15:39.071 | WARNING | Task run 'task_get_stars-225' - 'get_stars' task IP address: 172.18.0.21
16:15:39.073 | INFO    | Task run 'task_get_stars-

16:15:39.604 | INFO    | prefect.flow_runs.runner - Process for flow run 'prophetic-gaur' exited cleanly.

In [8]:
from prefect.settings import PREFECT_UI_URL
print(f"""
########
# NOTE #
########

Don't use the internal domain from the logs above: {PREFECT_UI_URL.value()!r}, use the public domain instead: {os.environ['PREFECT_PUBLIC']}
""")


########
# NOTE #
########

Don't use the internal domain from the logs above: 'http://prefect-server:4200', use the public domain instead: http://localhost:4200



In [13]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

15:36:52.877 | WARNING | root - Client IP address: 172.18.0.21

<div class="alert alert-info" role="alert">
Notes:

  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.
  1. Check in the run logs that the client IP address is also used by the Prefect flow and tasks.

### Deploy flows with Python

See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/deploy-via-python

Prefect offers a flexible way to deploy flows to dynamic infrastructure using the Python SDK. This approach allows you to target specific work pools and utilize dynamically provisioned infrastructure.

This is easier to deploy than with YAML (see next section) but less complete (e.g. cannot run additional scripts or pip install ...)

**Deploy the source code**

You want to deploy flows and tasks from you local source code... but this source code doesn't exist in the prefect worker node. 

So you need to store it somewhere and transfer it. 

We can use this project git repository but this is not very flexible (as for now you cannot even specify a git branch when deploying from python).

Another solution is to transfer the source code via the S3 bucket using prefect blocks.

In [9]:
if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Copy local source code to: {S3_BLOCK.basepath!r}")

# Use a subfolder named after the current user
s3_folder = f"code/{getpass.getuser()}" 

# Copy local directory contents
await S3_BLOCK.put_directory(local_path = ".", to_path = s3_folder)

# It doesn't follow symlinks so copy them manually
await S3_BLOCK.put_directory(local_path = "./resources", to_path = f"{s3_folder}/resources")

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Copy local source code to: 's3://prefect-share'


26

In [18]:
# Deploy the flow
flow = await prefect.flow.from_source(
    source=S3_BLOCK,
    entrypoint=f"{s3_folder}/my_prefect.py:flow_show_stars",
)
await flow.deploy(
    name="deploy-python-git",
    work_pool_name=PREFECT_WORK_POOL,
    tags=["tutorial"],
    ignore_warnings=True,
)
name = "flow-show-stars/deploy-python-git"
await my_prefect.wait_for_deployment(name)

Output()

Successfully created/updated all deployments!

                       Deployments                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ Name                              ┃ Status  ┃ Details ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩
│ flow-show-stars/deploy-python-git │ applied │         │
└───────────────────────────────────┴─────────┴─────────┘

To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'flow-show-stars/deploy-python-git'

You can also run your flow via the Prefect UI: http://prefect-server:4200/deployments/deployment/5233dcb1-15bf-4afc-bc14-9ce84b7896ae

Finished deploying prefect flow: 'flow-show-stars/deploy-python-git'


In [20]:
%%bash -s "$name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2"

Creating flow run for deployment 'flow-show-stars/deploy-python-git'...
Created flow run 'curious-toucan'.
└── UUID: a3a1ff37-8bfd-484a-8cea-6f1c1fd4136f
└── Parameters: {'github_repos': ['PrefectHQ/prefect', 'pydantic/pydantic', 'huggingface/transformers']}
└── Job Variables: {}
└── Scheduled start time: 2025-01-24 16:14:14 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/a3a1ff37-8bfd-484a-8cea-6f1c1fd4136f


In [14]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

15:38:37.783 | WARNING | root - Client IP address: 172.18.0.21

<div class="alert alert-info" role="alert">
Notes:

  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.
  1. Check in the run logs that the client IP address is **different** than the one used by the Prefect flow and tasks.

### Define deployments with YAML
See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/prefect-yaml

Use YAML to schedule and trigger flow runs and manage your code and deployments.

This is the most complete way to deploy your flow and tasks.

**Deploy from git repository**

As above, we need to store and transfer our flow and tasks from our local source code, saved in the project git repository. 

We can tell prefect to pull the source code from there before deploying it:
```yaml
pull:
- prefect.deployments.steps.git_clone:
    repository: https://github.com/org/repo.git
    branch: main
    credentials: "{{ prefect.blocks.github-credentials.my-credentials }}"
```
See the full yaml file: [deploy-yaml-git.yaml](./deploy-yaml-git.yaml)


In [45]:
filename = "deploy-yaml-git"
name = f"flow-show-stars/{filename}"

In [94]:
%%bash -s "$filename" "$PREFECT_WORK_POOL"
# Deploy the flow
prefect --no-prompt deploy --prefect-file "./$1.yaml" --pool "$2"

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'flow-show-stars/deploy-yaml-git' successfully created with id    │
│ 'ec4cede8-0cc0-4ec8-bade-6b927f443b4f'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/ec4cede8-0cc0-4ec8-bade-6b927f443b4f


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'flow-show-stars/deploy-yaml-git'



In [95]:
await my_prefect.wait_for_deployment(name)

Finished deploying prefect flow: 'flow-show-stars/deploy-yaml-git'


In [96]:
%%bash -s "$name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2"

Creating flow run for deployment 'flow-show-stars/deploy-yaml-git'...
Created flow run 'sophisticated-toucan'.
└── UUID: 9dcd6621-9275-43c5-bccc-f60a81dd1375
└── Parameters: {'github_repos': ['PrefectHQ/prefect', 'pydantic/pydantic', 'huggingface/transformers']}
└── Job Variables: {}
└── Scheduled start time: 2025-01-24 17:18:32 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/9dcd6621-9275-43c5-bccc-f60a81dd1375
